In [17]:
#jupyter notebook
import sqlite3
import random
import time
import requests
import pandas as pd
random.seed(42)

In [18]:
conn = sqlite3.connect("school.db")
cur = conn.cursor()
cur.execute("""
CREATE TABLE students (
student_id INTEGER PRIMARY KEY,
name TEXT, major TEXT, gpa REAL
)""")
cur.execute("""
CREATE TABLE enrollments (
enrollment_id INTEGER PRIMARY KEY,
student_id INTEGER, course TEXT, grade TEXT,
FOREIGN KEY (student_id) REFERENCES students(student_id)
)""")
# ... populate with sample rows (full code in the demo notebook) ...
conn.commit()

OperationalError: table students already exists

In [ ]:
query = """
SELECT name, gpa
FROM students
WHERE major = 'CS' AND gpa >= 3.5
ORDER BY gpa DESC
"""
print(pd.read_sql(query, conn))

Empty DataFrame
Columns: [name, gpa]
Index: []


In [ ]:
query = """
SELECT students.name, enrollments.course, enrollments.grade
FROM students
INNER JOIN enrollments ON students.student_id = enrollments.student_id
"""
print(pd.read_sql(query, conn))

Empty DataFrame
Columns: [name, course, grade]
Index: []


In [ ]:
query = """
SELECT major, COUNT(*) AS n_students, AVG(gpa) AS avg_gpa
FROM students
GROUP BY major
HAVING COUNT(*) > 2
ORDER BY avg_gpa DESC
"""
print(pd.read_sql(query, conn))

Empty DataFrame
Columns: [major, n_students, avg_gpa]
Index: []


In [ ]:
# UNSAFE — never build SQL with an f-string from user input
unsafe_query = "SELECT name, gpa FROM students WHERE major = '{unsafe_name}'"
# SAFE — the ? placeholder lets sqlite3 escape the value correctly
cur.execute("SELECT name, gpa FROM students WHERE major = ?", ("CS",))
rows = cur.fetchall()

In [ ]:
query = """
SELECT major, AVG(gpa) AS avg_gpa
FROM students
GROUP BY major
"""
df = pd.read_sql(query, conn)
print(df)
conn.close() # same discipline as closing a file

In [19]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
"latitude": 37.5665, # Seoul
"longitude": 126.9780,
"current_weather": True,
}
try:
  response = requests.get(url, params=params, timeout=10)
  print("status code:", response.status_code) # 200 = success
  data = response.json()
  print(data["current_weather"])
except requests.exceptions.RequestException:
# falls back to a sample response if the room has no Wi-Fi
  data = {"current_weather": {"temperature": 18.4, "windspeed": 9.1}}

status code: 200
{'time': '2026-10-08T01:15', 'interval': 900, 'temperature': 16.7, 'windspeed': 2.2, 'winddirection': 99, 'is_day': 1, 'weathercode': 0}


In [20]:
weather_df = pd.DataFrame([data["current_weather"]])
print(weather_df)

               time  interval  temperature  windspeed  winddirection  is_day  \
0  2026-10-08T01:15       900         16.7        2.2             99       1   

   weathercode  
0            0  


In [21]:
from bs4 import BeautifulSoup
url = "http://books.toscrape.com"
headers = {"User-Agent": "KDU-DataScience-Student-Demo"}
response = requests.get(url, headers=headers, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")
titles = [h.get_text(strip=True) for h in soup.select("h3 a")]
print(titles[:5])
time.sleep(1) # be polite before any next request

['A Light in the ...', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History ...']
